### 1. Carga de Datos y Exploración Inicial
Cargamos el dataset `heart_disease_health_indicators_BRFSS2015.csv` utilizando pandas y realizamos una exploración inicial para entender su estructura, tipos de datos y posibles valores faltantes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

# Cargar el dataset
file_path = '/content/heart_disease_health_indicators_BRFSS2015.csv'
df = pd.read_csv(file_path)

# Mostrar las primeras 5 filas del dataset
print("Primeras 5 filas del dataset:")
display(df.head())

# Mostrar información general del dataset (tipos de datos, valores no nulos)
print("\nInformación del dataset:")
df.info()

# Mostrar estadísticas descriptivas
print("\nEstadísticas descriptivas del dataset:")
display(df.describe())

### 2. Preprocesamiento de Datos
Este paso incluye la separación de características y la variable objetivo, la división del dataset en conjuntos de entrenamiento, validación (desarrollo) y prueba, y el escalado de las características numéricas.

In [ ]:
# Separar características (X) y la variable objetivo (y)
# Asumiendo que 'HeartDiseaseorAttack' es la variable objetivo (0: No, 1: Sí)
X = df.drop('HeartDiseaseorAttack', axis=1)
y = df['HeartDiseaseorAttack']

# Dividir los datos en conjuntos de entrenamiento (60%), validación (20%) y prueba (20%)
# Primero, dividimos en entrenamiento y un conjunto temporal (validación + prueba)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# Luego, dividimos el conjunto temporal en validación y prueba
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_val: {X_val.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")

# Escalado de características numéricas
# Es crucial escalar los datos para redes neuronales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrames para facilitar el análisis posterior (opcional, pero útil)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\nPrimeras 5 filas de X_train escalado:")
display(X_train_scaled.head())

### 3. Construcción de la Red Neuronal Multicapa (MLP) y Técnicas para Mitigar el Sobreajuste
Vamos a construir una red neuronal densa (MLP) utilizando Keras/TensorFlow. Incorporaremos varias técnicas para mitigar el sobreajuste:
-   **Capas Dropout**: Desactivan aleatoriamente un porcentaje de neuronas durante el entrenamiento, obligando a la red a aprender representaciones más robustas.
-   **Regularización L2 (Weight Decay)**: Añade una penalización a los pesos de la red, desalentando que los pesos tomen valores muy grandes.
-   **Early Stopping**: Detiene el entrenamiento cuando el rendimiento en el conjunto de validación deja de mejorar, evitando que el modelo aprenda ruido del conjunto de entrenamiento.

In [ ]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

# Compilar el modelo
# Usamos 'Adam' como optimizador, 'binary_crossentropy' como función de pérdida para clasificación binaria,
# y 'accuracy' como métrica principal.
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()

### 4. Entrenamiento del Modelo y Monitoreo del Sobreajuste
Entrenamos la red neuronal utilizando los datos de entrenamiento y validación. Usaremos `EarlyStopping` para detener el entrenamiento si el modelo comienza a sobreajustarse al conjunto de entrenamiento.

In [ ]:
# Definir la llamada de Early Stopping
# Monitorea la 'val_loss' (pérdida en el conjunto de validación)
# 'patience=10' significa que esperará 10 épocas sin mejora antes de detenerse
# 'restore_best_weights=True' restaurará los pesos del modelo de la mejor época
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_val_scaled, y_val), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")

### 5. Análisis del Rendimiento del Modelo y Visualización del Sobreajuste
Después de entrenar el modelo, evaluamos su rendimiento en el conjunto de prueba y visualizamos las curvas de pérdida y precisión de entrenamiento y validación para identificar señales de sobreajuste.

In [ ]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))

In [ ]:
# Mostrar la matriz de confusión
print("\nMatriz de Confusión en el conjunto de prueba:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.show()


In [ ]:
# Visualizar la historia del entrenamiento para detectar sobreajuste
# Curvas de pérdida
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Pérdida de Entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida de Validación')
plt.title('Curva de Pérdida')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()

In [ ]:
# Curvas de precisión
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Precisión de Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Precisión de Validación')
plt.title('Curva de Precisión')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()

plt.tight_layout()
plt.show()

print("\nAnálisis de las curvas:")
print(" - Si la 'Pérdida de Entrenamiento' disminuye mientras la 'Pérdida de Validación' comienza a aumentar, es una señal clara de sobreajuste.")
print(" - La 'Precisión de Validación' que se estanca o disminuye mientras la 'Precisión de Entrenamiento' sigue mejorando también indica sobreajuste.")
print(" - Gracias a Early Stopping, el entrenamiento se detuvo antes de que el sobreajuste fuera severo, y los mejores pesos se restauraron, lo que se refleja en un buen rendimiento en la validación.")